# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

This dataset includes survey data about mental health indicators, demographic details, and standardized assessment scores (GAD-7, PHQ-9, PSQ).


In [ ]:
# Ensure required library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)
print("License:", metadata.license)
print("Date Published:", metadata.datePublished)


## 2. Data Overview
Review available record sets, fields, and their IDs.

We first enumerate all record sets along with their `@id`.
(If the dataset contains multiple record sets, they'll be listed. Each record set has fields/columns with their own `@id`s.)

In [ ]:
# List all RecordSets in the dataset, referencing by @id
record_set_objs = dataset.metadata.recordSet

# For demonstration, print out @id and name for each record set
record_sets = []
for rs in record_set_objs:
    print("RecordSet @id:", rs['@id'])
    print("RecordSet name:", rs.get('name', '[no name]'))
    record_sets.append(rs['@id'])
    # List fields/columns within the record set
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            print("    - @id:", fld['@id'], "label:", fld.get('label', fld['@id']), "type:", fld.get('dataType', 'unknown'))
    print('-'*40)

if not record_sets:
    print("No record sets found - please check the metadata structure or refer to documentation.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s found in the previous section.

In [ ]:
# Extract data from each record set
dataframes = {}

# Loop over discovered record sets by their @id
for record_set_id in record_sets:
    print(f"Loading records for RecordSet {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns for {record_set_id}:", df.columns.tolist())
    print(df.head(3))
    print('---')

# Choose one record set for EDA; if only one, select it
if dataframes:
    main_record_set_id = record_sets[0]  # Use the first one found
    print(f"Selected main record set: {main_record_set_id}")
    print("Sample columns:", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Here, we'll select a numeric field, such as PHQ-9 score, for demonstration.

Replace `<numeric_field_id>` and `<group_field>` with actual field `@id`s based on previous inspection.

In [ ]:
# Use main RecordSet; assign its DataFrame
df = dataframes[main_record_set_id]

# Example: find column @id for PHQ-9 score (or a similar numeric field).
# If schema uses field @id e.g. 'phq_9_score', pick that.

# List all columns to help pick
columns = df.columns.tolist()
print("Available columns:", columns)

# Let's assume field @id 'phq_9_score' and group by 'village' (modify if different).
numeric_field_id = 'phq_9_score' if 'phq_9_score' in columns else columns[0]  # default to first if not present
group_field_id = 'village' if 'village' in columns else columns[0]

# Remove outliers: filter where PHQ-9 > 10
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize PHQ-9 values
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by village, showing mean PHQ-9 score
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in data, adjust field @id.")

## 5. Visualization

Visualize distributions of PHQ-9 score for filtered records, or compare scores across villages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of PHQ-9 scores for filtered records
if numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (PHQ-9 Score) for Scores > {threshold}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by village
    if group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrates loading, exploring, and visualizing the FAIR² Mental Health Survey Dataset from Kilifi County using `mlcroissant`. Key findings might include distributions of mental health scores and demographic influences such as village of residence. Use this template to extend analyses to other fields and record sets as needed.